In [1]:
# !pip -q install  fusion_solar_py pvlib retry-requests openmeteo_requests requests-cache python-telegram-bot

In [2]:
# !pip -q install git+https://github.com/EmaR97/EnergyManagementRL.git@testing-10

In [3]:
import os

from dotenv import load_dotenv

load_dotenv()

def get_env(name):
    return os.environ.get(name)

In [4]:
# from kaggle_secrets import UserSecretsClient
# 
# user_secrets = UserSecretsClient()
# 
# def get_env(name):
#     return user_secrets.get_secret(name)

In [5]:
FUSION_SOLAR_CLIENT_PASSWORD = get_env("FUSION_SOLAR_CLIENT_PASSWORD")
FUSION_SOLAR_CLIENT_USERNAME = get_env("FUSION_SOLAR_CLIENT_USERNAME")
LAT = float(get_env("LAT"))
LON = float(get_env("LON"))
TOKEN = get_env("TELEGRAM_BOT_TOKEN_TEST")
# TOKEN = get_env("TELEGRAM_BOT_TOKEN")
ADMIN_ID = int(get_env("TELEGRAM_ID"))

In [6]:
from energymanagementrl.utility import get_logger

logger_main = get_logger("Main")
logger_esm = get_logger("ESM")
logger_tb = get_logger("TB")

In [7]:
from energymanagementrl.production_forecast import *

panel_model = PanelModel(pdc0=0.42, temp_model_a=-3.56, temp_model_b=-0.075, delta_t=3, gamma_pdc=-0.004)
num_panels = 14
arrays = [ArrayConfig(name='sud_east', panel_model=panel_model, num_panels=num_panels, tilt_angle=25, azimuth=110),
          ArrayConfig(name='nord_west', panel_model=panel_model, num_panels=num_panels, tilt_angle=18, azimuth=290)]

_plant_config = PlantConfig(
    latitude=LAT, longitude=LON, timezone='Europe/Rome', inverter_pdc0=6, arrays=arrays
)
_production_forecaster = EnergyPredictionSystem(plant_config=_plant_config, open_meteo_client=OpenMeteoClient())

In [8]:
from energymanagementrl.fusion_solar_connector import *

_client = FusionSolarClientParsed(FUSION_SOLAR_CLIENT_USERNAME, FUSION_SOLAR_CLIENT_PASSWORD,
                                  huawei_subdomain="uni004eu5")
periodic_task = PeriodicTask(_client.keep_alive)
periodic_task.start()
_plant_id = _client.get_plant_ids()[0]
battery_id = _client.get_battery_ids(_plant_id)[0]

Periodic task started.


In [9]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np
from energymanagementrl.rl import load_model_with_weights

# Create a simple dummy environment used only to define the model struct, as the model will be used in inference
env = gym.Env()
env.action_space = spaces.Discrete(2)  # Two possible actions: 0 or 1
env.observation_space = spaces.Box(low=0, high=1000, shape=(55,), dtype=np.float64)  # 55 state variables

# Load the DQN policy
best_model = '../data/trained_models/models/dqn_1.0_0.06_0.06_0.02_1000_l_2.policy_weights.pth'
_model = load_model_with_weights(
    env,
    best_model,
    # 'dqn.2024_12_31_11_44_29.latest'
)

In [10]:
from energymanagementrl.rl.real_system_interaction import EnergyManagementSystem

system = EnergyManagementSystem(
    client=_client,
    plant_id=_plant_id,
    battery_id=battery_id,
    production_forecaster=_production_forecaster,
    model=_model,
    logger=logger_esm,
)

# system.set_active(True)

In [11]:
from energymanagementrl.interface import TelegramBot

bot = TelegramBot(
    system=system,
    token=TOKEN,
    allowed_users=[ADMIN_ID],
    logger=logger_tb,
)

2025-02-20 08:08:33,368 - TB - INFO - Handlers have been set up.
2025-02-20 08:08:33,369 - TB - INFO - TelegramBot initialized.


In [12]:
import asyncio
import nest_asyncio

nest_asyncio.apply()
loop = asyncio.get_event_loop()
loop.run_until_complete(bot.set_bot_commands())

2025-02-20 08:08:33,393 - TB - INFO - Setting bot commands...
2025-02-20 08:08:33,582 - TB - INFO - Bot commands set!


In [ ]:
import json
import logging
from contextlib import contextmanager

try:
    import pandas as pd
except ImportError:
    raise ImportError("Please install pandas: pip install pandas")

try:
    from kaggle_secrets import UserSecretsClient
except ImportError:
    raise ImportError("This script must run in a Kaggle environment with kaggle_secrets available.")

DATASET_NAME = "logs-persistence-test"
DATASET_FILE = "logs.csv"
METADATA_FILE = "dataset-metadata.json"
VERSION_MESSAGE = "Updated dataset with duplicated last row"
DELETE_OLD_VERSIONS = False

# Get Kaggle credentials
user_secrets = UserSecretsClient()
kaggle_key = user_secrets.get_secret("KAGGLE_KEY")
kaggle_username = user_secrets.get_secret("KAGGLE_USERNAME")

# Create .kaggle directory
kaggle_config_dir = os.path.expanduser('~/.kaggle')
os.makedirs(kaggle_config_dir, exist_ok=True)

# Write kaggle.json
kaggle_json_path = os.path.join(kaggle_config_dir, 'kaggle.json')
with open(kaggle_json_path, 'w') as f:
    json.dump({"username": kaggle_username, "key": kaggle_key}, f)
os.chmod(kaggle_json_path, 0o600)

try:
    import kaggle
    from kaggle import api as kaggle_api
except ImportError:
    raise ImportError("Please install kaggle: pip install kaggle")

kaggle.api.authenticate()

DATASET_ID = f"{kaggle_username}/{DATASET_NAME}"
DATASET_DIR = f"./{DATASET_NAME}"


def download_dataset(dataset_id: str, output_dir: str, file_path: str):
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    if not os.path.exists(os.path.join(output_dir, file_path)):
        print(f"Downloading dataset {dataset_id}...")
        kaggle_api.dataset_download_files(dataset_id, path=output_dir, unzip=True)
        kaggle_api.dataset_metadata(dataset_id, path=output_dir)
        print("Download and extraction completed.\n")


def update_metadata(file_path: str, dataset_id: str, output_dir: str):
    try:
        with open(os.path.join(output_dir, file_path), "r") as f:
            metadata = json.load(f)
        # If it's a stringified JSON, parse it; otherwise leave it as-is.
        if isinstance(metadata, str):
            metadata = json.loads(metadata)
        metadata["id"] = dataset_id
        with open(os.path.join(output_dir, file_path), "w") as f:
            json.dump(metadata, f, indent=4)
    except Exception as e:
        print(f"Metadata update error: {e}")


def upload_new_version(directory: str, message: str, delete_old: bool):
    try:
        kaggle_api.dataset_create_version(directory, message, delete_old_versions=delete_old)
    except Exception as e:
        print(f"Upload error: {e}")


class KaggleDatasetHandler(logging.Handler):
    log_df: pd.DataFrame

    def emit(self, record):
        try:
            entry = {
                'asctime': datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                'name': record.name,
                'levelname': record.levelname,
                'message': record.getMessage()
            }
            self.log_df.loc[len(self.log_df)] = entry
        except Exception as e:
            print(f"Logging error: {e}")

    def __init__(self, _log_df):
        super().__init__()
        self.log_df = _log_df


@contextmanager
def logger_to_kaggle_dataset(logger, dataset_id, dataset_dir, dataset_file, metadata_file):
    filename = os.path.join(dataset_dir, datetime.now().strftime("%Y%m%d") + "_" + dataset_file)

    handler = None  # <-- Add this
    handler_added = False  # <-- And this

    try:
        download_dataset(dataset_id, dataset_dir, metadata_file)

        if not os.path.exists(filename):
            log_df = pd.DataFrame(columns=['asctime', 'name', 'levelname', 'message'])
            log_df.to_csv(filename, index=False)
        else:
            log_df = pd.read_csv(filename)

        if not any(isinstance(h, KaggleDatasetHandler) for h in logger.handlers):
            handler = KaggleDatasetHandler(log_df)
            logger.addHandler(handler)
            handler_added = True

        yield logger
    finally:
        if handler is not None:  # <-- Important safety check
            handler.log_df.to_csv(filename, index=False)

        update_metadata(metadata_file, dataset_id, dataset_dir)
        upload_new_version(dataset_dir, VERSION_MESSAGE, DELETE_OLD_VERSIONS)

        if handler_added:
            logger.removeHandler(handler)


In [13]:
import threading

async def run_control_loop():
    with logger_to_kaggle_dataset(logger_esm, DATASET_ID, DATASET_DIR, DATASET_FILE, METADATA_FILE) as log:
        await system.control_loop()

threading.Thread(target=lambda: asyncio.run(run_control_loop())).start()

2025-02-20 08:08:33,601 - ESM - INFO - Starting Control loop


In [14]:
from datetime import datetime


async def shutdown():
    await sleep_async()
    await stopping_all()


async def stopping_all():
    logger_main.warning("Stopping all...")
    await system.stop_control_loop()
    bot.app.stop_running()


async def sleep_async(hour: int = None, minute: int = None):
    hour = hour or 11 if datetime.now().hour < 12 else 23
    minute = minute or 55
    now = datetime.now()
    stop_time = now.replace(hour=hour, minute=minute, second=0, microsecond=0)
    seconds = (stop_time - now).total_seconds()
    logger_main.info(f"Scheduled sleep until {stop_time}, sleeping for {seconds} seconds")
    await asyncio.sleep(seconds)
    logger_main.info(f"Scheduled sleep completed")


loop.create_task(
    shutdown()
    # sleep_async(7,51)
)

<Task pending name='Task-5' coro=<shutdown() running at /tmp/ipykernel_6301/1680221435.py:4>>

In [15]:
loop.run_until_complete(bot.run())

2025-02-20 08:08:33,800 - TB - INFO - Bot is starting...
2025-02-20 08:08:51,454 - TB - INFO - Authorization check for user 5541452419.
2025-02-20 08:08:51,455 - TB - INFO - User 5541452419 entered an unknown command: /set_controller_active
2025-02-20 08:09:31,587 - TB - INFO - Authorization check for user 5541452419.
2025-02-20 08:09:32,109 - TB - INFO - User 5541452419 requested system stats: {'battery_mode': 'MAXIMUM_SELF_CONSUMPTION', 'soc': '18.0%', 'to_battery': '0.225kw', 'from_battery': '0kw', 'prod': '0.563kw', 'load': '0.323kw', 'to_grid': '0.015kw', 'from_grid': '0kw'}.
2025-02-20 08:10:06,311 - TB - INFO - Authorization check for user 5541452419.
2025-02-20 08:10:06,312 - TB - INFO - User 5541452419 entered an unknown command: /set_controller_active
2025-02-20 08:10:36,919 - TB - INFO - Authorization check for user 5541452419.
2025-02-20 08:10:36,920 - TB - INFO - User 5541452419 requested command description
2025-02-20 08:21:54,085 - TB - INFO - Authorization check for use